# Goals

Determine how the bug described in https://github.com/IQSS/dataverse.harvard.edu/issues/500 can adversley affect search results of users of Harvard Dataverse and other Dataverse repositories.

# Plan

- Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

- For Harvard Dataverse:
    - Get the list of URLs and names of RORs in the externalvocabularyvalue table

    - Get the list of author affiliations with RORs; the list and count of datasets that have those author affiliations; and if each ROR is in the externalvocabularyvalue table
    - For each author affiliation with a ROR:
        - Search 1: Use the Search API to search for datasets by the name of the author affiliation
        - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
        - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

- For other Dataverse installations:
    - Determine which Dataverse installations have datasets with ROR URLs in their Author Affiliations or Funding Information Agencies
    - For each installation:

        - Get the list of author affiliations with RORs and the list and count of datasets that have those author affiliations (I don't know what's in each installation's externalvocabularyvalue table, so I can't include if each ROR is in the externalvocabularyvalue table)
        - For each author affiliation with a ROR:
            - Search 1: Use the Search API to search for datasets by the name of the author affiliation
            - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
            - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

# Import libraries and set variables

In [1]:
import csv
import json
import os
from os import listdir
import pandas as pd
from pathlib import Path
import platform
import re
import requests
import sys
import time
from time import sleep
from tqdm import tqdm
# from tqdm.auto import tqdm
tqdm_bar_format = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

In [ ]:
# Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation 
# fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER



allDataverseJSONExportFilesDirectory = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/all_json_metadata_files'

# From allDataverseJSONExportFilesDirectory, get the first 10 JSON files whose names include "(latest_version)" and save them to a new directory on my desktop
import os
import shutil

destinationDirectory = '/Users/juliangautier/Desktop/latest_version_json_files'
os.makedirs(destinationDirectory, exist_ok=True)

for root, dirs, files in os.walk(allDataverseJSONExportFilesDirectory):
    for file in files:
        if "(latest_version)" in file:
            shutil.copy(os.path.join(root, file), destinationDirectory)
            if len(os.listdir(destinationDirectory)) >= 50:
                break
    if len(os.listdir(destinationDirectory)) >= 50:
        break

# Get all names, ROR URLs in Author Affiliation and Funding Information Agency fields
Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields in datasets published in all known Dataverse installations. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

In [ ]:
# Load and query the Dataverse JSON exports of the latest version of each dataset. If the export includes ROR URLs in the author affiliation 
# or funding information agency field, save the dataset PID, the field name (author affiliation or funding information agency), the ROR name, and the ROR URL

filePath = '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataverse_json_metadata_from_each_known_dataverse_installation_unzipped/CORA._Research_Data_Repository_(RDR)_2026.09.02_11.05.39/Dataverse_JSON_metadata_2026.09.02_11.05.39/doi_10.34810_DATA3521_v1.0(latest_version).json'
with open(filePath, 'r') as f1:
    datasetMetadata = f1.read()
    datasetMetadata = json.loads(datasetMetadata)

citationBlock = datasetMetadata['data']['metadataBlocks']['citation']
for field in citationBlock['fields']:
    if field['typeName'] == 'author':
        authors = field['value']

for author in authors:
    authorAffiliationName = author['authorAffiliation']['expandedvalue']['termName']
    authorAffiliationRor = author['authorAffiliation']['value']

    print(f'{authorAffiliationName} ({authorAffiliationRor})')

# print(json.dumps(authors, indent=4))

authorsDf = pd.read_csv(
    'author(citation)_2026.08.25-2026.09.02.csv',
    # parse_dates=['dataset_publication_date'],
    usecols=lambda x: x in ['dataset_pid', 'dataset_pid_url', 'authorAffiliation'],
    # dtype={'latest_dataset_version_number': str},
    encoding='utf-8-sig'
)

renamedColumns = {'authorAffiliation': 'author_affiliation'}
authorsDf = authorsDf.rename(columns=renamedColumns)

# Create new dataframe with only authors where there's a ROR in the authorAffiliation field, using regex to match the ROR pattern (https://ror.org/ followed by 9 alphanumeric characters)
authorsWithRORAffiliationsDf = authorsDf[authorsDf['author_affiliation'].str.contains(r'https://ror\.org/[0-9a-zA-Z]{9}', regex=True, na=False)]

print(f'Count of rows in authorsWithRORAffiliationsDf: {len(authorsWithRORAffiliationsDf)}')

authorsWithRORAffiliationsDf.info()

Use the dataset PID and my dataset_pids_from_most_known_dataverse_installations.csv file to indicate which installation each dataset PID is in, so that I can group the PIDs by Dataverse installation

In [ ]:
objectColumns = [
    'dataverse_installation_name',
    'dataset_pid',
    'dataset_pid_url']
typeDict = {col: str for col in objectColumns}

datasetsByInstallationsDf = pd.read_csv(
    '/Users/juliangautier/Documents/all_installation_metadata_2026.08.25_22.57.59/dataset_pids_from_most_known_dataverse_installations.csv',
    usecols=lambda x: x in ['dataverse_installation_name', 'dataset_pid', 'dataset_pid_url'],
    dtype=typeDict,
    encoding='utf-8-sig'
)

# Measure bug's affect on findability of datasets in Harvard Dataverse

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in Harvard Dataverse

# Measure bug's affect on findability of datasets in other Dataverse installations

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in each Dataverse installation